In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize

BASE = Path.cwd().parent
DATA = BASE / "Datasets" / "processed" / "dataset.csv"
RESULTS = BASE / "Results"

df = pd.read_csv(DATA).fillna({"title": "", "abstract": "", "ref_dois": ""})
label_cols = [c for c in df.columns if c.startswith("lbl_")]
names = [c[4:] for c in label_cols]
Y = df[label_cols].values


dev_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20,
                                     random_state=42, shuffle=True)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.25,
                                      random_state=42, shuffle=True)
fit_idx = np.concatenate([train_idx, val_idx])

print(f"train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")

train 4914 | val 1639 | test 1639


In [2]:

text = (df.title + " " + df.abstract).values
vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2,
                      sublinear_tf=True, stop_words="english")
X_text_fit = vec.fit_transform(text[fit_idx])
X_text_test = vec.transform(text[test_idx])


vocabulary, rows, cols = {}, [], []
for i, refs in enumerate(df.ref_dois):
    for r in (x for x in refs.split("|") if x):
        j = vocabulary.setdefault(r, len(vocabulary))
        rows.append(i); cols.append(j)
X_cite = normalize(csr_matrix((np.ones(len(rows)), (rows, cols)),
                              shape=(len(df), len(vocabulary))))

print(f"text features:     {X_text_fit.shape[1]:,}")
print(f"citation features: {X_cite.shape[1]:,}")


def evaluate(pred):
    return (f1_score(Y[test_idx], pred, average="micro", zero_division=0),
            f1_score(Y[test_idx], pred, average="macro", zero_division=0))


def to_labels(scores, threshold):
    pred = (scores >= threshold).astype(int)
    empty = pred.sum(1) == 0
    pred[empty, scores[empty].argmax(1)] = 1
    return pred


results = {}

text features:     50,000
citation features: 367,569


In [3]:


for tag, Xf, Xt in [("TEXT   (BR + SVM)", X_text_fit, X_text_test),
                    ("CITE   (BR + SVM)", X_cite[fit_idx], X_cite[test_idx])]:
    m = OneVsRestClassifier(LinearSVC(C=2.0, max_iter=3000), n_jobs=-1)
    m.fit(Xf, Y[fit_idx])
    mi, ma = evaluate(to_labels(m.decision_function(Xt), 0.0))
    results[tag] = (mi, ma)
    print(f"{tag:<22} micro-F1 {mi:.3f}   macro-F1 {ma:.3f}")

TEXT   (BR + SVM)      micro-F1 0.750   macro-F1 0.745
CITE   (BR + SVM)      micro-F1 0.678   macro-F1 0.668


In [4]:


S = (X_cite[test_idx] @ X_cite[fit_idx].T).toarray()
k = 20
nn = np.argpartition(-S, k, axis=1)[:, :k]

scores_knn = np.zeros((len(test_idx), Y.shape[1]))
for i in range(len(test_idx)):
    w = S[i, nn[i]]
    if w.sum() > 0:
        scores_knn[i] = (Y[fit_idx][nn[i]] * w[:, None]).sum(0) / w.sum()

mi, ma = evaluate(to_labels(scores_knn, 0.35))
results["CITE   (BC-kNN k=20)"] = (mi, ma)
print(f"{'CITE   (BC-kNN k=20)':<22} micro-F1 {mi:.3f}   macro-F1 {ma:.3f}")

CITE   (BC-kNN k=20)   micro-F1 0.752   macro-F1 0.750


In [5]:



X_both_fit = hstack([X_text_fit, X_cite[fit_idx]]).tocsr()
X_both_test = hstack([X_text_test, X_cite[test_idx]]).tocsr()

m = OneVsRestClassifier(LinearSVC(C=2.0, max_iter=3000), n_jobs=-1)
m.fit(X_both_fit, Y[fit_idx])
mi, ma = evaluate(to_labels(m.decision_function(X_both_test), 0.0))
results["HYBRID (concatenation)"] = (mi, ma)
print(f"{'HYBRID (concatenation)':<22} micro-F1 {mi:.3f}   macro-F1 {ma:.3f}")


def zscale(a):
    return (a - a.mean()) / (a.std() + 1e-9)

m_text = OneVsRestClassifier(LinearSVC(C=2.0, max_iter=3000), n_jobs=-1)
m_text.fit(X_text_fit, Y[fit_idx])
s_text = m_text.decision_function(X_text_test)

w, thr = 0.35, 1.0       
combined = (1 - w) * zscale(s_text) + w * zscale(scores_knn)
mi, ma = evaluate(to_labels(combined, thr))
results["HYBRID (score-level)"] = (mi, ma)
print(f"{'HYBRID (score-level)':<22} micro-F1 {mi:.3f}   macro-F1 {ma:.3f}")

HYBRID (concatenation) micro-F1 0.758   macro-F1 0.753
HYBRID (score-level)   micro-F1 0.779   macro-F1 0.778


In [6]:
print(f"{'model':<26}{'micro-F1':>10}{'macro-F1':>10}")
print("-" * 46)
for tag, (mi, ma) in results.items():
    print(f"{tag:<26}{mi:>10.3f}{ma:>10.3f}")

pd.DataFrame([{"model": k, "micro_f1": round(v[0], 3), "macro_f1": round(v[1], 3)}
              for k, v in results.items()]).to_csv(
    RESULTS / "model_comparison.csv", index=False)
print(f"\nsaved to {RESULTS / 'model_comparison.csv'}")

model                       micro-F1  macro-F1
----------------------------------------------
TEXT   (BR + SVM)              0.750     0.745
CITE   (BR + SVM)              0.678     0.668
CITE   (BC-kNN k=20)           0.752     0.750
HYBRID (concatenation)         0.758     0.753
HYBRID (score-level)           0.779     0.778

saved to /Users/kalai/Documents/Dissertation/Results/model_comparison.csv


In [7]:

print(f"{'field':<20}{'train':>8}{'val':>8}{'test':>8}")
for j, n in enumerate(names):
    print(f"{n:<20}"
          f"{int(Y[train_idx][:, j].sum()):>8}"
          f"{int(Y[val_idx][:, j].sum()):>8}"
          f"{int(Y[test_idx][:, j].sum()):>8}")

test_counts = Y[test_idx].sum(axis=0)
print(f"\nsmallest field in test set: {int(test_counts.min())} papers "
      f"({names[int(test_counts.argmin())]})")
print(f"largest field in test set:  {int(test_counts.max())} papers "
      f"({names[int(test_counts.argmax())]})")

field                  train     val    test
Biotechnology            604     191     205
Chemistry                984     332     340
CS_AI                    600     211     224
CS_InfoSystems           614     201     196
Engineering_EE           853     291     291
Environmental            601     194     193
MaterialsScience         813     288     283
Mathematics              625     192     183
Physics_Applied          586     206     212
Psychology               595     217     203

smallest field in test set: 183 papers (Mathematics)
largest field in test set:  340 papers (Chemistry)


In [8]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfTransformer


vocabulary, rows, cols = {}, [], []
for i, refs in enumerate(df.ref_dois):
    for r in (x for x in refs.split("|") if x):
        j = vocabulary.setdefault(r, len(vocabulary))
        rows.append(i); cols.append(j)
R_raw = csr_matrix((np.ones(len(rows)), (rows, cols)),
                   shape=(len(df), len(vocabulary)))


doc_freq = np.asarray((R_raw > 0).sum(axis=0)).ravel()

n = len(df)
print(f"total references: {len(doc_freq):,}")
print(f"cited by exactly 1 paper:  {(doc_freq == 1).sum():,}")
print(f"cited by 2+ papers:        {(doc_freq >= 2).sum():,}")
print(f"cited by >1% of papers:    {(doc_freq > 0.01 * n).sum():,}")
print(f"most-cited reference appears in {doc_freq.max()} papers")

total references: 367,569
cited by exactly 1 paper:  301,149
cited by 2+ papers:        66,420
cited by >1% of papers:    28
most-cited reference appears in 440 papers


In [9]:
def knn_scores(X, fit, score, k=30):
    S = (X[score] @ X[fit].T).toarray()
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    out = np.zeros((len(score), Y.shape[1]))
    for i in range(len(score)):
        w = S[i, nn[i]]
        if w.sum() > 0:
            out[i] = (Y[fit][nn[i]] * w[:, None]).sum(0) / w.sum()
    return out

def svm_pred(X, fit, score):
    m = OneVsRestClassifier(LinearSVC(C=2.0, max_iter=3000), n_jobs=-1)
    m.fit(X[fit], Y[fit])
    return m.predict(X[score])

f1 = lambda idx, p: f1_score(Y[idx], p, average="micro", zero_division=0)

variants = {
    "no pruning":                 (np.ones(len(doc_freq), bool), False),
    "remove singletons":          (doc_freq >= 2, False),
    "singletons + common (>1%)":  ((doc_freq >= 2) & (doc_freq <= 0.01 * n), False),
    "singletons + IDF weighting": (doc_freq >= 2, True),
}

print(f"{'variant':<28}{'features':>10}{'kNN val':>9}{'kNN test':>10}"
      f"{'SVM val':>9}{'SVM test':>10}")
pruning_rows = []
for name, (keep, use_idf) in variants.items():
    X = R_raw[:, keep]
    X = (TfidfTransformer(norm="l2").fit_transform(X) if use_idf
         else normalize(X))

    kv = f1(val_idx,  to_labels(knn_scores(X, train_idx, val_idx), 0.35))
    kt = f1(test_idx, to_labels(knn_scores(X, fit_idx, test_idx), 0.35))
    sv = f1(val_idx,  svm_pred(X, train_idx, val_idx))
    st = f1(test_idx, svm_pred(X, fit_idx, test_idx))

    print(f"{name:<28}{X.shape[1]:>10,}{kv:>9.3f}{kt:>10.3f}{sv:>9.3f}{st:>10.3f}")
    pruning_rows.append({"variant": name, "features": X.shape[1],
                         "knn_val": kv, "knn_test": kt,
                         "svm_val": sv, "svm_test": st})

pd.DataFrame(pruning_rows).round(3).to_csv(RESULTS / "citation_pruning.csv", index=False)

variant                       features  kNN val  kNN test  SVM val  SVM test
no pruning                     367,569    0.760     0.753    0.418     0.470
remove singletons               66,420    0.763     0.760    0.594     0.631
singletons + common (>1%)       66,392    0.771     0.761    0.592     0.627
singletons + IDF weighting      66,420    0.773     0.763    0.588     0.623


In [10]:
from sklearn.feature_extraction.text import TfidfTransformer


vocabulary, rows, cols = {}, [], []
for i, refs in enumerate(df.ref_dois):
    for r in (x for x in refs.split("|") if x):
        j = vocabulary.setdefault(r, len(vocabulary))
        rows.append(i); cols.append(j)
R_raw = csr_matrix((np.ones(len(rows)), (rows, cols)),
                   shape=(len(df), len(vocabulary)))


doc_freq = np.asarray((R_raw > 0).sum(axis=0)).ravel()

n = len(df)
print(f"total references: {len(doc_freq):,}")
print(f"cited by exactly 1 paper:  {(doc_freq == 1).sum():,}")
print(f"cited by 2+ papers:        {(doc_freq >= 2).sum():,}")
print(f"cited by >1% of papers:    {(doc_freq > 0.01 * n).sum():,}")
print(f"most-cited reference appears in {doc_freq.max()} papers")

total references: 367,569
cited by exactly 1 paper:  301,149
cited by 2+ papers:        66,420
cited by >1% of papers:    28
most-cited reference appears in 440 papers


In [11]:
def knn_scores(X, fit, score, k=20):
    S = (X[score] @ X[fit].T).toarray()
    nn = np.argpartition(-S, k, axis=1)[:, :k]
    out = np.zeros((len(score), Y.shape[1]))
    for i in range(len(score)):
        w = S[i, nn[i]]
        if w.sum() > 0:
            out[i] = (Y[fit][nn[i]] * w[:, None]).sum(0) / w.sum()
    return out

def svm_pred(X, fit, score):
    m = OneVsRestClassifier(LinearSVC(C=2.0, max_iter=3000), n_jobs=-1)
    m.fit(X[fit], Y[fit])
    return to_labels(m.decision_function(X[score]), 0.0)

f1 = lambda idx, p: f1_score(Y[idx], p, average="micro", zero_division=0)

variants = {
    "no pruning":                 (np.ones(len(doc_freq), bool), False),
    "remove singletons":          (doc_freq >= 2, False),
    "singletons + common (>1%)":  ((doc_freq >= 2) & (doc_freq <= 0.01 * n), False),
    "singletons + IDF weighting": (doc_freq >= 2, True),
}

print(f"{'variant':<28}{'features':>10}{'kNN val':>9}{'kNN test':>10}"
      f"{'SVM val':>9}{'SVM test':>10}")
pruning_rows = []
for name, (keep, use_idf) in variants.items():
    X = R_raw[:, keep]
    X = (TfidfTransformer(norm="l2").fit_transform(X) if use_idf
         else normalize(X))

    kv = f1(val_idx,  to_labels(knn_scores(X, train_idx, val_idx), 0.35))
    kt = f1(test_idx, to_labels(knn_scores(X, fit_idx, test_idx), 0.35))
    sv = f1(val_idx,  svm_pred(X, train_idx, val_idx))
    st = f1(test_idx, svm_pred(X, fit_idx, test_idx))

    print(f"{name:<28}{X.shape[1]:>10,}{kv:>9.3f}{kt:>10.3f}{sv:>9.3f}{st:>10.3f}")
    pruning_rows.append({"variant": name, "features": X.shape[1],
                         "knn_val": kv, "knn_test": kt,
                         "svm_val": sv, "svm_test": st})

pd.DataFrame(pruning_rows).round(3).to_csv(RESULTS / "citation_pruning.csv", index=False)
print(f"\nsaved to {RESULTS / 'citation_pruning.csv'}")

variant                       features  kNN val  kNN test  SVM val  SVM test
no pruning                     367,569    0.760     0.752    0.651     0.678
remove singletons               66,420    0.766     0.761    0.692     0.704
singletons + common (>1%)       66,392    0.771     0.760    0.694     0.705
singletons + IDF weighting      66,420    0.773     0.760    0.694     0.704

saved to /Users/kalai/Documents/Dissertation/Results/citation_pruning.csv


In [12]:

grid = []
for k in [5, 10, 20, 30, 50]:
    s = knn_scores(X_cite, train_idx, val_idx, k)
    for t in [0.20, 0.25, 0.30, 0.35, 0.40]:
        grid.append({"k": k, "threshold": t,
                     "val_micro_f1": f1_score(Y[val_idx], to_labels(s, t),
                                              average="micro", zero_division=0)})

g = pd.DataFrame(grid).pivot(index="k", columns="threshold",
                             values="val_micro_f1").round(4)
print(g)
g.to_csv(RESULTS / "knn_tuning_grid.csv")

threshold    0.20    0.25    0.30    0.35    0.40
k                                                
5          0.7279  0.7409  0.7467  0.7520  0.7484
10         0.7323  0.7472  0.7553  0.7551  0.7580
20         0.7310  0.7445  0.7555  0.7605  0.7579
30         0.7267  0.7417  0.7543  0.7601  0.7591
50         0.7263  0.7406  0.7532  0.7569  0.7535


In [13]:

thr = 0.35
Yt = Y[test_idx]
max_score = scores_knn.max(axis=1)
pred = to_labels(scores_knn, thr)
fallback = max_score < thr
exact = (pred == Yt).all(axis=1)
top_ok = Yt[np.arange(len(Yt)), scores_knn.argmax(1)] == 1

print(f"test papers: {len(Yt)}")
print(f"fallback rule used (max score < {thr}): {fallback.sum()} ({100 * fallback.mean():.1f}%)")
print(f"  of which max score = 0 (no coupled neighbours): {(max_score == 0).sum()}")

print(f"\n{'group':<16}{'papers':>8}{'exact match':>13}{'top field correct':>19}")
for name, m in [("threshold met", ~fallback), ("fallback", fallback)]:
    print(f"{name:<16}{m.sum():>8}{exact[m].mean():>13.3f}{top_ok[m].mean():>19.3f}")

print("\nFallback cases by maximum score:")
for lo, hi, lab in [(0, 1e-12, "exactly 0"), (1e-12, 0.1, "0 to 0.1"),
                    (0.1, 0.2, "0.1 to 0.2"), (0.2, thr, f"0.2 to {thr}")]:
    m = fallback & (max_score >= lo) & (max_score < hi)
    acc = f"{top_ok[m].mean():.3f}" if m.sum() else "  -"
    print(f"  {lab:<12} {m.sum():>5} papers   top field correct {acc}")

print(f"\nIf papers below a confidence level were marked 'uncertain':")
print(f"{'uncertain if max <':<20}{'coverage':>10}{'micro-F1 on the rest':>22}")
rows = []
for tau in [0.0, 0.1, 0.2, 0.3, 0.35, 0.5, 0.7]:
    keep = max_score >= tau
    f = f1_score(Yt[keep], pred[keep], average="micro", zero_division=0)
    rows.append({"tau": tau, "coverage": keep.mean(), "micro_f1": f})
    print(f"{tau:<20}{100 * keep.mean():>9.1f}%{f:>22.3f}")
pd.DataFrame(rows).round(3).to_csv(RESULTS / "uncertainty_coverage.csv", index=False)

test papers: 1639
fallback rule used (max score < 0.35): 27 (1.6%)
  of which max score = 0 (no coupled neighbours): 20

group             papers  exact match  top field correct
threshold met       1612        0.554              0.800
fallback              27        0.296              0.333

Fallback cases by maximum score:
  exactly 0       20 papers   top field correct 0.300
  0 to 0.1         0 papers   top field correct   -
  0.1 to 0.2       0 papers   top field correct   -
  0.2 to 0.35      7 papers   top field correct 0.429

If papers below a confidence level were marked 'uncertain':
uncertain if max <    coverage  micro-F1 on the rest
0.0                     100.0%                 0.752
0.1                      98.8%                 0.755
0.2                      98.8%                 0.755
0.3                      98.7%                 0.755
0.35                     98.4%                 0.756
0.5                      92.6%                 0.773
0.7                      67.8%